In [10]:
import asyncio
import csv
import json
import time
from urllib.parse import urlparse
from urllib.robotparser import RobotFileParser

from playwright.async_api import async_playwright

In [13]:
class SiteCrawler:
    def __init__(self, url_queue: list=[], politeness_delay: int=2):
        """
        :param url_queue: list of URLs to crawl
        :param politeness_delay: seconds to wait between requests to the SAME domain
        """
        self.url_queue = url_queue
        self.politeness_delay = politeness_delay
        self.url_content = {url: "" for url in self.url_queue}
        self._robots_cache = {}       # domain -> RobotFileParser
        self._last_request_time = {}  # domain -> timestamp of last request

    def _get_domain(self, url):
        return urlparse(url).netloc

    def _is_allowed(self, url, user_agent="*"):
        """Check robots.txt for this URL's domain, caching parsers per domain."""
        domain = self._get_domain(url)
        if domain not in self._robots_cache:
            rp = RobotFileParser()
            robots_url = f"{urlparse(url).scheme}://{domain}/robots.txt"
            try:
                rp.set_url(robots_url)
                rp.read()
            except Exception:
                # If robots.txt is unreachable/missing, default to allow
                rp = None
            self._robots_cache[domain] = rp

        rp = self._robots_cache[domain]
        if rp is None:
            return True
        return rp.can_fetch(user_agent, url)

    async def _wait_politely(self, url):
        """Respect politeness_delay on a per-domain basis (not global)."""
        domain = self._get_domain(url)
        last = self._last_request_time.get(domain)
        if last is not None:
            elapsed = time.monotonic() - last
            remaining = self.politeness_delay - elapsed
            if remaining > 0:
                await asyncio.sleep(remaining)
        self._last_request_time[domain] = time.monotonic()

    async def get_content(self, url, browser=None):
        """
        Fetch and cache page content for a single URL.
        :param url: page to fetch
        :param browser: an already-launched playwright browser instance to reuse;
                         if None, a temporary one is launched just for this call
        """
        if url in self.url_content and self.url_content[url]:
            return self.url_content[url]  # already crawled this session

        if not self._is_allowed(url):
            self.url_content[url] = ""
            return ""

        await self._wait_politely(url)

        own_browser = browser is None
        if own_browser:
            playwright = await async_playwright().start()
            browser = await playwright.chromium.launch()

        try:
            page = await browser.new_page()
            try:
                await page.goto(url, wait_until="networkidle", timeout=30000)
                html = await page.content()
            except Exception as e:
                print(f"Failed to fetch {url}: {e}")
                html = ""
            finally:
                await page.close()
        finally:
            if own_browser:
                await browser.close()
                await playwright.stop()

        self.url_content[url] = html
        return html

    async def parse_queue(self, max_concurrency: int = 4):
        """
        Crawl everything currently in the queue, reusing one browser instance.
        Politeness delay is enforced per-domain, so different domains can
        still be fetched concurrently while same-domain requests are spaced out.
        """
        async with async_playwright() as p:
            browser = await p.chromium.launch()
            semaphore = asyncio.Semaphore(max_concurrency)

            async def fetch_one(url):
                async with semaphore:
                    await self.get_content(url, browser=browser)

            await asyncio.gather(*(fetch_one(url) for url in self.url_queue))
            await browser.close()

        self.url_queue = []

    def add_url_to_queue(self, url):
        """
        :param url:
        :return:
        """
        if url in self.url_content or url in self.url_queue:
            return
        self.url_queue.append(url)
        self.url_content[url] = ""

    def load_url_csv(self, path: str):
        """
        Load URLs from a single-column CSV (no header assumed) and add
        any not already known to the queue.
        :param path:
        :return:
        """
        with open(path, "r", newline="") as f:
            reader = csv.reader(f)
            for row in reader:
                if not row:
                    continue
                url = row[0].strip()
                if url:
                    self.add_url_to_queue(url)

    def load_content_json(self, path: str):
        """
        Load a previously saved {url: content} mapping, merging into
        self.url_content. Existing non-empty entries are not overwritten.
        :param path:
        :return:
        """
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for url, content in data.items():
            if url not in self.url_content or not self.url_content[url]:
                self.url_content[url] = content

    def save_url_csv(self, path: str, save_type="w"):
        """
        :param path:
        :param save_type:
        :return:
        """
        save_string = ""
        for url in self.url_queue:
            save_string += url + "\n"
        with open(path, save_type) as f:
            f.write(save_string)

    def save_content_json(self, path: str, save_type="w"):
        """
        :param path:
        :param save_type:
        :return:
        """
        if save_type == "a":
            # merge with existing file content rather than overwriting
            try:
                with open(path, "r", encoding="utf-8") as f:
                    existing = json.load(f)
            except (FileNotFoundError, json.JSONDecodeError):
                existing = {}
            existing.update(self.url_content)
            data_to_write = existing
        else:
            data_to_write = self.url_content

        with open(path, "w", encoding="utf-8") as f:
            json.dump(data_to_write, f, ensure_ascii=False, indent=2)

In [17]:
crawler = SiteCrawler()
crawler.load_url_csv("../starting_urls.csv")
await crawler.parse_queue()
content = await crawler.get_content("http://www.pro-igel.de/")

Failed to fetch https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-dresden-chemnitz-leipzig-sachsen: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-dresden-chemnitz-leipzig-sachsen", waiting until "networkidle"

Failed to fetch https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-schleswig-holstein-bremen-hamburg: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-schleswig-holstein-bremen-hamburg", waiting until "networkidle"

Failed to fetch https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-berlin-brandenburg-mecklenburg-vorpommern: Page.goto: Timeout 30000ms exceeded.
Call log:
  - navigating to "https://www.wildtierschutz-deutschland.de/wildtier-notfall/wildtierauffangstationen-berlin-br

In [19]:

single_crawler = SiteCrawler("http://www.mauersegler.com/")
print(f"Crawler with queue {single_crawler.url_queue} created.")
await single_crawler.parse_queue()
print("Crawl complete!")
content = await single_crawler.get_content("http://www.mauersegler.com/")
print(f"Content: {content}")

Crawler with queue http://www.mauersegler.com/ created.
Failed to fetch h: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to "h", waiting until "networkidle"

Failed to fetch t: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to "t", waiting until "networkidle"

Failed to fetch p: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to "p", waiting until "networkidle"

Failed to fetch :: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to ":", waiting until "networkidle"

Failed to fetch t: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to "t", waiting until "networkidle"

Failed to fetch w: Page.goto: Protocol error (Page.navigate): Cannot navigate to invalid URL
Call log:
  - navigating to "w", waiting until "networkidle"

Failed to fetc

In [20]:
from bs4 import BeautifulSoup
def clean_html(html:str) -> str:
    #TODO: Add options and docs
    soup = BeautifulSoup(html, 'html.parser')

    for tag in soup(["script", "style", "noscript", "svg", "iframe", "nav", "footer", "header", "form"]):
        tag.decompose() # Remove the specified tags TODO: Put in config file

    text = soup.get_text(separator="\n")
    lines = [line.strip() for line in text.splitlines() if line.strip()]
    return "\n".join(lines)

print(clean_html(content))

Startseite - Deutsche Gesellschaft für Mauersegler e.V.
Zum Hauptinhalt springen
Wir helfen Mauerseglern in Not.
Beratung,
medizinische Versorgung
und Schutzprojekte
seit 32 Jahren.
Vogel gefunden?
Jetzt unterstützen
Notfall-Hotline
+49 (69) 35 35 15 04
Was wir tun
Von der Rettung bis
zur Auswilderung.
Beratung
Rund-um-die-Uhr-Beratung für alle, die einen verletzten oder verwaisten Mauersegler finden.
Mehr erfahren →
Klinik
Spezialisierte tiermedizinische Versorgung für Mauersegler in unserer Klinik in Frankfurt.
Mehr erfahren →
Auswilderung
Nach erfolgreicher Genesung begleiten wir Mauersegler zurück in die Wildnis.
Mehr erfahren →
Schutzprojekte
Lebensraumschutz, Nisthilfen und Aufklärungsarbeit für den Mauerseglerbestand.
Mehr erfahren →
Warum das wichtig ist
Jahrzehnte Engagement,
in Zahlen.
0+
Pfleglinge
Aufgenommen, behandelt und entlassen seit 1994.
24/7
Hotline
Rund um die Uhr erreichbar.
365
Tage im Jahr
Geöffnet an jedem einzelnen Tag.
0+
Jahre Erfahrung
Expertise, aufgebaut 